<a href="https://colab.research.google.com/github/Phani-ISB/Reconciliation_Modules/blob/main/2_bank_reconciliation%20Rev.%2001_04042026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import all required libraries

In [ ]:
# Install prerequisities
!pip install -q rapidfuzz

In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta

# Tool to generate all possible unique pairs/groups
from itertools import combinations
# To calculate matching score between two strings [score : 0-100]
from rapidfuzz import fuzz


# Data Ingestion

In [ ]:
# Loading excel files of Ledger and Bank statement

ledger_df = pd.read_excel('/content/Ledger_InputData.xlsx')
bank_df = pd.read_excel('/content/Statement_InputData.xlsx')

In [ ]:
# Standardizing column names
ledger_df.columns = ledger_df.columns.str.strip()
bank_df.columns = bank_df.columns.str.strip()

print("Ledger Shape:", ledger_df.shape)
print("Bank Shape:", bank_df.shape)

Ledger Shape: (2977, 10)
Bank Shape: (250, 11)


# EDA

In [ ]:
# Summary of Ledger
ledger_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Doc Type            2938 non-null   object        
 1   Posting Dt          2977 non-null   datetime64[ns]
 2   Doc No              2977 non-null   int64         
 3   Cheque No           14 non-null     float64       
 4   Description         1268 non-null   object        
 5   Particulars         832 non-null    object        
 6   Debit-Credit        2977 non-null   float64       
 7   Bank                2946 non-null   object        
 8   GL A/C              2969 non-null   float64       
 9   GL A/c Description  2977 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 232.7+ KB


In [ ]:
#Summary of Bank statement
bank_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          250 non-null    datetime64[ns]
 1   Unique ID     215 non-null    object        
 2   Bank Ref      249 non-null    object        
 3   Cust Ref      249 non-null    object        
 4   Narration     249 non-null    object        
 5   Narration 2   249 non-null    object        
 6   Opening Flag  1 non-null      object        
 7   Credit-Debit  250 non-null    float64       
 8   BANK          250 non-null    object        
 9   Credit        34 non-null     float64       
 10  Debit         215 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 21.6+ KB


In [ ]:
# Create unified amount columns in both ledger and bank statement (keeping same symbol +, -)
ledger_df['Amount'] = ledger_df['Debit-Credit'].fillna(0)
bank_df['Amount'] = bank_df['Credit-Debit'].fillna(0)

In [ ]:
#  Understanding Debit/Credit in Ledger & Statement
print("\n Debits & Credits in LEDGER")
print("Ledger Debit Total:", ledger_df[ledger_df['Amount'] > 0]['Amount'].sum())
print("Ledger Credit Total:", ledger_df[ledger_df['Amount'] < 0]['Amount'].sum())
print("Ledger balances against Debit/Credit:", ledger_df['Amount'].sum())


print("\n Debits & Credits in STATEMENT")
print("Bank Debit Total:", bank_df[bank_df['Amount'] < 0]['Amount'].sum())
print("Bank Credit Total:", bank_df[bank_df['Amount'] > 0]['Amount'].sum())
print("Bank balances against Debit/Credit:", bank_df['Amount'].sum())

print("\n Overall Difference:", ledger_df['Amount'].sum() - bank_df['Amount'].sum())


 Debits & Credits in LEDGER
Ledger Debit Total: 116410182.16
Ledger Credit Total: -123096062.35779999
Ledger balances against Debit/Credit: -6685880.19780001

 Debits & Credits in STATEMENT
Bank Debit Total: -32271013.099999998
Bank Credit Total: 31409973.31
Bank balances against Debit/Credit: -861039.7899999991

 Overall Difference: -5824840.407800011


In [ ]:
# Normalize narration text from both Ledger and Bank statement
ledger_df['Narration'] = (ledger_df['Description'].astype(str) + " " + ledger_df['Particulars'] + " " + ledger_df['GL A/c Description']).str.lower().str.strip()
bank_df['Narration'] = (bank_df['Narration'].astype(str) + " " + bank_df['Narration 2'].astype(str)).str.lower().str.strip()

In [ ]:
# Duplicates Check
print("Ledger Duplicates:", ledger_df.duplicated().sum())
print("Bank Duplicates:", bank_df.duplicated().sum())

Ledger Duplicates: 378
Bank Duplicates: 0


In [ ]:
# Duplicates data
duplicate_rows = ledger_df[ledger_df.duplicated(keep=False)]

# Sort them so the identical rows are next to each other
duplicate_rows.sort_values(by=list(ledger_df.columns)).head(10)

,Doc Type,Posting Dt,Doc No,Cheque No,Description,Particulars,Debit-Credit,Bank,GL A/C,GL A/c Description,Amount,Narration
1910,KZ,2021-01-12,601017,NaN,Salary,PAID AGAINST EE EXP REIMBURSE,-145.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-145.0,salary paid against ee exp reimburse accounts ...
1911,KZ,2021-01-12,601017,NaN,Salary,PAID AGAINST EE EXP REIMBURSE,-145.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-145.0,salary paid against ee exp reimburse accounts ...
33,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary paid against ee expense reimb accounts ...
34,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary paid against ee expense reimb accounts ...
35,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary paid against ee expense reimb accounts ...
38,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary paid against ee expense reimb accounts ...
1683,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary paid against ee expense reimb accounts ...
30,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary paid against ee expense reimb accounts ...
37,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary paid against ee expense reimb accounts ...
848,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary paid against ee expense reimb accounts ...


# Reconciliation by Rules

In [ ]:
# Initialise matching columns in both the dataframes (ledger and bank statement)
# Initialise tolerances on date , amount and fuzzy logic (narration matching)
ledger_df['Matched'] = False
bank_df['Matched'] = False

ledger_df['GroupID'] = None
bank_df['GroupID'] = None

ledger_df['Comment'] = None
bank_df['Comment'] = None

DATE_TOLERANCE = 3
AMOUNT_TOLERANCE = 0.00
FUZZY_THRESHOLD = 70


In [ ]:
# Helper functions for rules

def amount_match(a, b):
    return abs(a - b) <= AMOUNT_TOLERANCE

def date_exact(d1, d2):
    return d1 == d2

def date_range(d1, d2):
    return abs((d1 - d2).days) <= DATE_TOLERANCE

def narration_exact(n1, n2):
    return n1 == n2

def narration_fuzzy(n1, n2):
    if pd.isna(n1) or pd.isna(n2):
        return False
    return fuzz.partial_ratio(n1, n2) >= FUZZY_THRESHOLD

In [ ]:
# Matching Duplicates (Internal Duplicate) across Ledger and Bank statement
def tag_internal_duplicates(df, drop_duplicates=False, keep='first',group_start=1):
    df = df.copy()

    # Create full row header data of all columns
    header = df.astype(str).agg('|'.join, axis=1)

    # Find duplicates
    dup_groups = header[header.duplicated(keep=False)]

    recon_id = group_start
    group_map = {}

    for key, idxs in dup_groups.groupby(dup_groups).groups.items():
        for i in idxs:
            group_map[i] = recon_id
        recon_id += 1

    # Assign values
    df['GroupID'] = df.index.map(group_map)
    df['Rule'] = df['GroupID'].apply(
        lambda x: "Duplicate Data" if pd.notna(x) else None
    )

    if drop_duplicates :
      df = df.loc[~header.duplicated(keep=keep)].copy()

    return df, recon_id

In [ ]:
# Defining Sequence of rules


rules = [
    {
        "name": "Narration Exact",
        "func": lambda l, b: narration_exact(l['Narration'], b['Narration'])
    },
    {
        "name": "Narration Exact/Fuzzy",
        "func": lambda l, b: (
            narration_exact(l['Narration'], b['Narration']) or
            narration_fuzzy(l['Narration'], b['Narration'])
        )
    },
    {
        "name": "Narration + Date Exact",
        "func": lambda l, b: (
            narration_exact(l['Narration'], b['Narration']) and
            date_exact(l['Posting Dt'], b['Date'])
        )
    },
    {
        "name": "Narration + Date Range",
        "func": lambda l, b: (
            narration_exact(l['Narration'], b['Narration']) and
            date_range(l['Posting Dt'], b['Date'])
        )
    },
    {
        "name": "Narration Fuzzy + Date Range",
        "func": lambda l, b: (
            (narration_exact(l['Narration'], b['Narration']) or
             narration_fuzzy(l['Narration'], b['Narration'])) and
            (date_exact(l['Posting Dt'], b['Date']) or
             date_range(l['Posting Dt'], b['Date']))
        )
    },
    {
        "name": "Date Exact",
        "func": lambda l, b: date_exact(l['Posting Dt'], b['Date'])
    },
    {
        "name": "Date Range",
        "func": lambda l, b: date_range(l['Posting Dt'], b['Date'])
    }

]

In [ ]:
# Compare both ledger and bank to find matching transactions based on rules
def run_rules(ledger_df, bank_df, rules, start_gid):
    # initialise empty list of matches
    matches = []
    gid = start_gid

    for rule in rules :
    # Check unmatched transactions and initiate them for rule matching
        l_un = ledger_df[~ledger_df['Matched']]
        b_un = bank_df[~bank_df['Matched']]
    # Looping every unmatched row in ledger against all rows in bank statement
    # li (index) and lrow (row data)
        for li, lrow in l_un.iterrows():
    # Potential candidates identified (first with amount check)
            candidates = b_un[
                abs(b_un['Amount'] - lrow['Amount']) <= AMOUNT_TOLERANCE
            ]
            if candidates.empty:
                continue
    # Candidates with specific logic of rule applied
            candidates = candidates[
                candidates.apply(lambda brow: rule["func"](lrow, brow), axis=1)
            ]

            if not candidates.empty:
                bi = candidates.index[0]

                # Mark matched
                ledger_df.at[li, 'Matched'] = True
                bank_df.at[bi, 'Matched'] = True

                # Assign GroupID
                ledger_df.at[li, 'GroupID'] = gid
                bank_df.at[bi, 'GroupID'] = gid

                # Assign Comment
                ledger_df.at[li, 'Comment'] = rule["name"]
                bank_df.at[bi, 'Comment'] = rule["name"]

                matches.append({
                    "Ledger_Index": li,
                    "Bank_Index": bi,
                    "Rule": rule["name"],
                    "GroupID": gid
                })

                gid += 1

    return matches, gid

In [ ]:
# One-to-Many / Many-to-One Matching combinations across statement and ledger
def subset_sum_match(df, target, tol):
    for r in range(2, min(6, len(df) + 1)):
        for combo in combinations(df.index, r):
            if abs(df.loc[list(combo), 'Amount'].sum() - target) <= tol:
                return list(combo)
    return None

In [ ]:
# Simplifying selections by symbol positive or negative values
def greedy_match(df, target, tol):
    total, selected = 0, []
    for idx, amt in df['Amount'].abs().sort_values(ascending=False).items():
        val = df.at[idx, 'Amount']
        if abs(total + val) <= abs(target) + tol:
            total += val
            selected.append(idx)
            if abs(total - target) <= tol: return selected
    return None

In [ ]:
# For reconciliation with one-to-many and many-to-one transactions against ledger and bank

def match_group(ledger_df, bank_df, gid, direction="Many-to-One"):
    matches = []
    # Identify which side is the 'Many' (Source) and which is the 'One' (Target)
    if direction == "Many-to-One":
        source_df, target_df = ledger_df, bank_df
    else:
        source_df, target_df = bank_df, ledger_df

    for ti, trow in target_df[~target_df['Matched']].iterrows():
        # Step 1: Filter potential candidates
        mask = (~source_df['Matched']) & \
               (source_df['Amount'] * trow['Amount'] > 0) & \
               (source_df['Amount'].abs() <= abs(trow['Amount']) + AMOUNT_TOLERANCE)

        candidates = source_df[mask]

        if 0 < len(candidates) <= 50:
            # Step 2: Run Greedy Match logic
            group = greedy_match(candidates, trow['Amount'], AMOUNT_TOLERANCE)

            if group:
                # Step 3: Update DataFrames in bulk
                source_df.loc[group, ['Matched', 'GroupID', 'Comment']] = [True, gid, direction]
                target_df.loc[ti, ['Matched', 'GroupID', 'Comment']] = [True, gid, direction]

                # Step 4: Standardize the output keys to avoid extra columns
                if direction == "Many-to-One":
                    l_idx, b_idx = group, [ti]  # Ledger is many, Bank is one
                else:
                    l_idx, b_idx = [ti], group  # Ledger is one, Bank is many

                matches.append({
                    "GroupID": gid,
                    "Rule": direction,
                    "Ledger_Indices": l_idx,
                    "Bank_Indices": b_idx
                })

                gid += 1

    return matches, gid

In [ ]:
all_matches = []
recon_id = 1


# Apply internal duplicacy rule on both dataframes
ledger_df, recon_id = tag_internal_duplicates(ledger_df, drop_duplicates=True, keep='first', group_start= recon_id )
bank_df, recon_id = tag_internal_duplicates(bank_df, drop_duplicates=True, keep='first', group_start= recon_id)

# Step 1: Run 1-to-1 rules
matches , recon_id = run_rules(ledger_df, bank_df, rules, recon_id)
all_matches += matches

# Step 2: Run Group Matching rules
matches , recon_id = match_group(ledger_df, bank_df, recon_id, direction = "Many-to-One")
all_matches += matches

matches , recon_id = match_group(ledger_df, bank_df, recon_id, direction = "One-to-Many")
all_matches += matches

In [ ]:
# Collect all the results
recon_df = pd.DataFrame(all_matches)
if 'Ledger_Indices' in recon_df.columns:
    # Fill NaN in Ledger_Index with the first value from Ledger_Indices
    recon_df['Ledger_Index'] = recon_df['Ledger_Index'].fillna(recon_df['Ledger_Indices'])
    recon_df['Bank_Index'] = recon_df['Bank_Index'].fillna(recon_df['Bank_Indices'])
recon_df = recon_df.explode('Ledger_Index').explode('Bank_Index')

# Drop the temporary columns
recon_df = recon_df.drop(columns=['Ledger_Indices', 'Bank_Indices'], errors='ignore')

# Reset index for a clean look
recon_df = recon_df.reset_index(drop=True)

In [ ]:
recon_df

,Ledger_Index,Bank_Index,Rule,GroupID
0,1871.0,23.0,Narration Exact/Fuzzy,22
1,1880.0,23.0,Narration Exact/Fuzzy,23
2,3.0,216.0,Date Exact,24
3,10.0,55.0,Date Exact,25
4,11.0,168.0,Date Exact,26
...,...,...,...,...
181,547,51,One-to-Many,199
182,946,91,One-to-Many,200
183,1879,49,One-to-Many,201
184,2093,217,One-to-Many,202


In [ ]:
match_summary = recon_df['Rule'].value_counts().reset_index()
match_summary.columns = ['Rule','Count']

In [ ]:
print(match_summary)

                    Rule  Count
0             Date Exact    162
1            One-to-Many     10
2            Many-to-One     10
3  Narration Exact/Fuzzy      2
4             Date Range      2


In [ ]:
# Saving the files
ledger_df.to_excel('reconciled_ledger.xlsx', index=False)
bank_df.to_excel('reconciled_bank.xlsx', index=False)